# Léxico y reglas: construir un analizador de valoración

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

## Pregunta

¿Cómo podemos asignar una valoración a un texto mediante reglas explícitas?

## Objetivo

Construir paso a paso un analizador que suma y resta puntos según las palabras de un texto, y averiguar dónde acierta y dónde falla.

## Al terminar este cuaderno vas a poder:

1. Explicar cómo cada palabra de un léxico suma o resta a la puntuación de un texto.
2. Distinguir una puntuación calculada de una valoración interpretada por una persona.
3. Reconocer qué casos (negación, ironía, valoración mixta) rompen una regla, con la tabla de desafíos como evidencia.
4. Proponer una modificación y contrastarla con ejemplos antes y después.

## Alcance

El léxico y las reglas no constituyen un instrumento validado. Los resultados sirven para estudiar el procedimiento, no para describir cómo opinan los estudiantes argentinos.

## ◈ Marco contextual

**Analogía.** Imaginá a alguien que corrige un texto con dos marcadores: verde para las palabras que suenan favorables y rojo para las desfavorables. Al final cuenta cuántas marcas hay de cada color y decide. Nuestro analizador hace lo mismo, pero con una lista de palabras escrita de antemano y con valores numéricos en lugar de colores.

**Bajada tecnológica.** Este enfoque se llama *análisis de sentimiento basado en léxico*. Fue la base de muchos analizadores anteriores al aprendizaje automático y todavía se usa cuando hace falta que cada decisión sea explicable palabra por palabra (por ejemplo, la herramienta VADER para redes sociales en inglés). En el cuaderno 02 vamos a ver cómo un programa puede aprender esa lista a partir de ejemplos, y en el 03 cómo se comporta un modelo ya entrenado por terceros.

## ◈ Microglosario

| Término | Qué significa en este cuaderno |
|---|---|
| **Token** | Cada pieza en que se corta un texto: una palabra o un signo de puntuación. |
| **Léxico** | Lista de palabras con un valor numérico: positivo si favorece, negativo si desfavorece. |
| **Puntuación** | Suma de los valores de todos los tokens de un texto. |
| **Etiqueta** | Categoría final: `POS` (positivo), `NEG` (negativo) o `NEU` (neutro). |
| **Etiqueta de referencia** | La etiqueta que propone el protocolo docente. Es la vara con la que comparamos, no una verdad. |
| **Regla** | Instrucción explícita que modifica el cálculo, como "si hay un *no* antes, invertí el valor". |

## 1. Configuración del entorno

**Operación.** Importar las herramientas y registrar con qué versiones estamos trabajando.

**Por qué importa.** Los resultados de un análisis dependen del entorno en el que se ejecuta. Dejar las versiones a la vista permite comparar y repetir el trabajo.

In [ ]:
# sys e importlib.metadata: para registrar qué versiones estamos usando.
import sys
import importlib.metadata

# re: expresiones regulares, para cortar el texto en tokens.
import re

# pandas: para trabajar con tablas. display: para mostrarlas completas en el cuaderno.
import pandas as pd
from IPython.display import display

# utils.py (en esta carpeta): ayudas que dibujan los gráficos. No calculan nada.
from utils import ETIQUETAS, dibujar_barras, dibujar_tokens_con_valor, dibujar_matriz_de_confusión

# Mostramos hasta 140 caracteres por celda para que los textos no se corten.
pd.set_option("display.max_colwidth", 140)

print("Python:", sys.version.split()[0])
for paquete in ["pandas", "scikit-learn", "matplotlib"]:
    print(f"{paquete}: {importlib.metadata.version(paquete)}")

**Leé el resultado.** Tienen que aparecer las tres versiones, sin errores. Si falta alguna, el entorno no está bien armado: volvé a `ANTES_DE_LA_CLASE.md`.

Este cuaderno no usa números aleatorios: cada vez que lo ejecutes, con los mismos datos, vas a obtener el mismo resultado.

## 2. El corpus

**Pregunta.** ¿Con qué textos vamos a probar el analizador?

**Ficha de los datos**

| Campo | Contenido |
|---|---|
| Fuente | Ejemplos construidos por el docente para esta actividad. No son testimonios reales. |
| Archivos | `datos/corpus_sentimiento.csv` (33 textos) y `datos/desafios.csv` (8 textos) |
| Unidad | Una oración corta sobre trámites, atención y servicios |
| Cobertura | 24 textos de *entrenamiento* y 9 de *prueba*, repartidos en partes iguales entre `POS`, `NEG` y `NEU`; más 8 desafíos sin etiqueta |
| Límites | Las etiquetas son propuestas docentes que requieren discusión. La prueba es deliberadamente fácil: comparte vocabulario con el entrenamiento. Los desafíos incluyen casos que a propósito el método no debería resolver bien. |
| Licencia | Material del curso, de uso educativo |

**Decisión.** Los desafíos quedan aparte y sin etiqueta de referencia automática: son para pensar, no para puntuar.

In [ ]:
# read_csv lee un archivo de tabla en texto plano y lo convierte en una tabla de pandas.
corpus = pd.read_csv("datos/corpus_sentimiento.csv")
desafíos = pd.read_csv("datos/desafios.csv")

# head() muestra las primeras filas, para confirmar que la lectura salió bien.
display(corpus.head())
display(desafíos)

**Verificación.** Antes de seguir, confirmamos que los datos cumplen lo que esperamos.

In [ ]:
# assert detiene el cuaderno con un mensaje si lo que afirmamos no se cumple.
assert corpus["id"].is_unique, "Cada texto debe tener un identificador único."
assert corpus["etiqueta_referencia"].isin(ETIQUETAS).all(), "Las etiquetas solo pueden ser NEG, NEU o POS."

# crosstab cuenta cuántos textos hay en cada combinación de partición y etiqueta.
conteo = pd.crosstab(index=corpus["partición"], columns=corpus["etiqueta_referencia"])
display(conteo)

**Leé la tabla.** Cada etiqueta tiene 8 textos de entrenamiento y 3 de prueba. El reparto parejo evita que un método parezca bueno solo porque acierta la clase más abundante.

La mezcla de valoraciones no equivale automáticamente a neutralidad. Los casos ambiguos deben conservar una nota explicativa.

**Criterio inicial del protocolo**

- `POS`: valoración favorable explícita.
- `NEG`: valoración desfavorable explícita.
- `NEU`: información sin valoración explícita según este protocolo.

### ✎ Antes de ejecutar

Leé estos fragmentos y respondé por escrito para cada uno, **antes** de ver qué hace el analizador:

1. La atención fue buena.
2. La atención no fue buena.
3. La atención no fue mala.
4. La profesora explicó bien, pero la plataforma funciona mal.

- ¿Qué objeto o situación se valora?
- ¿Qué expresión sostiene tu interpretación?
- ¿Asignarías `POS`, `NEG` o `NEU`?
- ¿La etiqueta pierde alguna información importante?

No uses todavía una salida automática para justificar tu lectura.

**Mis anotaciones**

| Fragmento | Objeto valorado | Etiqueta propuesta | Evidencia textual |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | información que se pierde con una sola etiqueta: |

## 3. El léxico: la lista de palabras con valor

**Pregunta.** ¿Cómo convertimos "esta palabra suena favorable" en un número?

**Operación.** Escribimos un *diccionario* de Python: una colección de pares `"palabra": valor`. Cada par se lee como una ficha: la palabra y los puntos que aporta.

In [ ]:
# Cada par es "palabra": valor. Positivo favorece; negativo desfavorece.
# Las personas que armaron esta lista decidieron cada valor.
léxico = {
    "bueno": 1, "buena": 1, "bien": 1, "excelente": 2,
    "sencillo": 1, "sencilla": 1, "útil": 1,
    "claro": 1, "clara": 1, "rápido": 1, "rápida": 1,
    "amable": 1, "conforme": 1,
    "malo": -1, "mala": -1, "mal": -1, "pésimo": -2, "pésima": -2,
    "complicado": -1, "complicada": -1, "inútil": -1,
    "confuso": -1, "confusa": -1, "lento": -1, "lenta": -1,
    "desagradable": -1, "disconforme": -1
}

# items() entrega cada par palabra-valor para armar una tabla de dos columnas.
tabla_léxico = pd.DataFrame(léxico.items(), columns=["palabra", "valor"])

dibujar_barras(
    tabla=tabla_léxico, columna_nombre="palabra", columna_valor="valor",
    título="Léxico: puntos que aporta cada palabra",
)

**Leé el gráfico.** Las barras azules suman, las naranjas restan. Hay 27 palabras: casi todas valen 1 o −1, y solo `excelente`, `pésimo` y `pésima` llegan a 2 o −2.

### Qué decidimos al construir el léxico

Los valores fueron asignados por personas.

- ¿Por qué `excelente` vale 2 y `buena` vale 1?
- ¿Tenemos evidencia de que esa distancia representa intensidad?
- ¿Qué palabras faltan?
- ¿Una palabra conserva el mismo valor en cualquier contexto?

Los números son parte de una convención de medición. No prueban por sí solos que una experiencia sea dos veces más positiva que otra.

## 4. Cortar el texto en tokens

**Operación.** Antes de buscar palabras en el léxico, hay que separarlas. La función `tokenizar` pasa el texto a minúsculas y lo corta con una *expresión regular*, una fórmula que describe qué piezas buscar:

- `\w+` encuentra palabras (letras y números seguidos).
- `[^\w\s]` encuentra signos: cualquier cosa que no sea letra ni espacio.

**Decisión 1.** Conservamos la puntuación porque después la usaremos para limitar el alcance de una regla.

**Decisión 2.** No eliminamos palabras vacías: palabras muy frecuentes como `no` pueden ser decisivas para la tarea.

**Límite.** No hace análisis sintáctico ni lematización: `buenísimo` y `bueno` son tokens distintos.

In [ ]:
def tokenizar(texto):
    """Corta un texto en tokens: palabras y signos, todo en minúsculas.

    Recibe un texto. Devuelve una lista de tokens.
    """
    texto_en_minúsculas = texto.lower()
    # findall devuelve todas las piezas que coinciden con la expresión regular.
    tokens = re.findall(pattern=r"\w+|[^\w\s]", string=texto_en_minúsculas)
    return tokens


texto_ejemplo = "La atención no fue buena, pero el trámite fue sencillo."
print(tokenizar(texto_ejemplo))

**Leé la lista.** Cada palabra y cada signo (`,` y `.`) quedó como un token separado, y todo está en minúsculas. Fijate que `no` y `pero` se conservaron: los vamos a necesitar.

## 5. Puntuar una oración, paso a paso

**Pregunta.** ¿Cómo se convierte una lista de tokens en una puntuación?

**Procedimiento.** Para cada token, buscamos su valor en el léxico. Si no está, vale 0. Después sumamos. Primero lo hacemos a mano, en una oración, para ver cada paso.

In [ ]:
tokens = tokenizar(texto_ejemplo)

filas = []
for token in tokens:
    # get busca el token en el léxico; si no está, devuelve 0.
    valor = léxico.get(token, 0)
    filas.append({"token": token, "valor": valor})

contribuciones = pd.DataFrame(filas)
puntuación_total = contribuciones["valor"].sum()

dibujar_tokens_con_valor(tabla=contribuciones, columna_valor="valor", título=texto_ejemplo)
print("Puntuación total:", puntuación_total)

**Leé el resultado.** Dos fichas tienen valor: `buena` (+1) y `sencillo` (+1). El resto vale 0. La puntuación total es +2.

- **Observación:** el analizador solo "ve" las palabras que están en el léxico.
- **Interpretación posible:** un 0 puede significar "no hay valoración", pero también "hay una valoración con palabras que el léxico no conoce".
- **Límite:** la suma no distingue de qué se habla. `buena` califica a la atención y `sencillo` al trámite, pero ambos puntos se mezclan en un solo número.

Ahora que el procedimiento está claro, lo encapsulamos en funciones con nombre para poder usarlo con cualquier texto.

In [ ]:
def contribuciones_léxicas(texto):
    """Busca el valor de cada token de un texto en el léxico.

    Recibe un texto. Devuelve una tabla con una fila por token y su valor.
    """
    filas = []
    for token in tokenizar(texto):
        valor = léxico.get(token, 0)
        filas.append({"token": token, "valor": valor})
    return pd.DataFrame(filas)


def etiqueta_desde_puntuación(puntuación):
    """Convierte una puntuación en etiqueta: POS si es mayor que 0, NEG si es menor, NEU si es 0."""
    if puntuación > 0:
        return "POS"
    if puntuación < 0:
        return "NEG"
    return "NEU"


def puntuación_básica(texto):
    """Suma los valores del léxico de un texto, sin ninguna regla adicional."""
    tabla = contribuciones_léxicas(texto)
    return int(tabla["valor"].sum())


def predecir_básico(texto):
    """Devuelve la etiqueta que le corresponde a un texto según el método básico."""
    puntuación = puntuación_básica(texto)
    return etiqueta_desde_puntuación(puntuación)

### ✎ Antes de ejecutar

El método básico solo suma. Anticipá qué puntuación y qué etiqueta le va a asignar a los cuatro fragmentos que anotaste antes:

1. La atención fue buena.
2. La atención no fue buena.
3. La atención no fue mala.
4. La profesora explicó bien, pero la plataforma funciona mal.

- ¿"La atención fue buena" y "La atención no fue buena" van a recibir la misma puntuación? ¿Por qué?
- ¿Qué parte del código explicaría el resultado?
- Si una puntuación da 0, ¿significa que no hay valoración? ¿Podría ser una compensación entre valoraciones opuestas?

Escribí tus predicciones y después ejecutá.

In [ ]:
pares = pd.DataFrame({"texto": [
    "La atención fue buena.",
    "La atención no fue buena.",
    "La atención no fue mala.",
    "La profesora explicó bien, pero la plataforma funciona mal."
]})

# apply ejecuta una función sobre cada fila de la columna y junta los resultados.
pares["puntuación"] = pares["texto"].apply(puntuación_básica)
pares["etiqueta"] = pares["puntuación"].apply(etiqueta_desde_puntuación)
display(pares)

**Leé la tabla.**

- **Observación:** "La atención fue buena" y "La atención no fue buena" reciben exactamente la misma puntuación (+1) y la etiqueta `POS`. "La atención no fue mala" recibe −1 y `NEG`. En la cuarta oración, +1 y −1 se compensan y la puntuación es 0, es decir `NEU`.
- **Interpretación posible:** `no` vale 0 en el léxico, así que el método no se entera de que invierte el sentido. Y un 0 acá no significa "sin valoración": significa dos valoraciones opuestas que se anulan.
- **Límite:** esto no permite decir que el método "entiende" las oraciones. Solo muestra que sumar valores no alcanza para estos casos.

Anotá si se cumplió tu expectativa y qué línea del código explica el resultado.

## 6. Primera regla: la negación

**Pregunta.** ¿Podemos arreglar el caso de "no fue buena" con una regla explícita?

**La regla, en palabras.** Para cada palabra del léxico, miramos hacia atrás unos pocos tokens. Si encontramos un negador (`no`, `nunca`, `jamás`) antes de toparnos con un signo o un conector que corte la frase, invertimos el signo del valor.

Tres parámetros definen la regla. Cambiarlos cambia qué textos se consideran negados:

| Parámetro | Qué es | Qué cambia si lo modificás |
|---|---|---|
| `NEGADORES` | Palabras que invierten el valor | Agregar `tampoco` hace que "tampoco fue buena" se lea como negación. |
| `LÍMITES` | Signos y conectores que cortan el alcance de la negación | Si sacás `pero`, un `no` anterior puede afectar lo que viene después del `pero`. |
| `VENTANA` | Cuántos tokens hacia atrás mira la regla | Con 1, solo detecta `no` pegado a la palabra; con 5, alcanza frases más largas pero también puede equivocarse más. |

**Excepción declarada.** Acá usamos `enumerate`, que entrega cada token junto con su número de posición, porque la regla necesita saber *dónde* está cada palabra para mirar hacia atrás.

In [ ]:
NEGADORES = {"no", "nunca", "jamás"}
LÍMITES = {".", ",", ";", ":", "!", "?", "¿", "¡", "pero", "aunque", "sin", "embargo"}
VENTANA = 3  # cuántos tokens hacia atrás mira la regla


def hay_negación_antes(tokens, posición):
    """Dice si la palabra en esa posición queda negada por un negador cercano.

    Recibe la lista de tokens y la posición de la palabra. Devuelve True si,
    mirando hacia atrás hasta VENTANA tokens, aparece un negador antes que un límite.
    """
    inicio = max(0, posición - VENTANA)
    anteriores = tokens[inicio:posición]

    # reversed recorre del token más cercano al más lejano.
    for token_anterior in reversed(anteriores):
        if token_anterior in LÍMITES:
            return False
        if token_anterior in NEGADORES:
            return True
    return False

**Segunda función.** Con `hay_negación_antes` armamos la tabla que da el valor final de cada token.

In [ ]:
def contribuciones_con_negación(texto):
    """Calcula el valor de cada token aplicando la regla de negación.

    Recibe un texto. Devuelve una tabla con el valor base (del léxico),
    si se detectó negación y el valor final.
    """
    tokens = tokenizar(texto)
    filas = []
    for posición, token in enumerate(tokens):
        valor_base = léxico.get(token, 0)

        negada = False
        if valor_base != 0:
            negada = hay_negación_antes(tokens=tokens, posición=posición)

        valor_final = valor_base
        if negada:
            valor_final = -valor_base

        filas.append({"token": token, "valor_base": valor_base,
                      "negación_detectada": negada, "valor_final": valor_final})
    return pd.DataFrame(filas)


tabla_negación = contribuciones_con_negación("La atención no fue buena.")
dibujar_tokens_con_valor(tabla=tabla_negación, columna_valor="valor_base", título="Sin la regla")
dibujar_tokens_con_valor(tabla=tabla_negación, columna_valor="valor_final", título="Con la regla de negación")

**Leé los dos gráficos.** `buena` pasó de +1 (azul) a −1 (naranja): la regla encontró `no` dentro de la ventana y no había ningún límite en el medio.

**Una celda más.** Completamos el circuito con las funciones que suman y asignan la etiqueta con la regla activa.

In [ ]:
def puntuación_con_negación(texto):
    """Suma los valores finales de un texto, con la regla de negación aplicada."""
    tabla = contribuciones_con_negación(texto)
    return int(tabla["valor_final"].sum())


def predecir_con_negación(texto):
    """Devuelve la etiqueta de un texto según el método con negación."""
    puntuación = puntuación_con_negación(texto)
    return etiqueta_desde_puntuación(puntuación)

### ✎ Antes de ejecutar

La regla no analiza la estructura de la oración: solo mira hacia atrás unos pocos tokens. Anticipá qué etiqueta va a asignar a cada caso y si coincide con tu lectura:

- La atención no fue buena.
- No creo que la atención haya sido buena.
- La atención no fue buena ni amable.
- La atención no fue mala.

Escribí tus predicciones y una justificación breve, y después ejecutá.

In [ ]:
casos_negación = pd.DataFrame({"texto": [
    "La atención no fue buena.",
    "No creo que la atención haya sido buena.",
    "La atención no fue buena ni amable.",
    "La atención no fue mala."
]})
casos_negación["método básico"] = casos_negación["texto"].apply(predecir_básico)
casos_negación["con negación"] = casos_negación["texto"].apply(predecir_con_negación)
display(casos_negación)

**Leé la tabla.**

| Caso | Observación | Interpretación posible |
|---|---|---|
| "no fue buena" | Pasó de `POS` a `NEG`. | La regla funciona en el caso más simple. |
| "No creo que… buena" | Sigue en `POS`. | `buena` queda a más de 3 tokens del `no`, fuera de la ventana. Probá `VENTANA = 5`, pero tené en cuenta que ampliar el alcance también puede dar falsos positivos. |
| "no fue buena ni amable" | Da `NEU`. | `buena` se niega (−1) pero `amable` queda fuera de la ventana (+1), y se compensan. |
| "no fue mala" | Pasó de `NEG` a `POS`. | La regla invierte el signo, pero "no fue mala" no equivale siempre a "fue buena": puede ser una valoración moderada. |

**Límite.** La regla no identifica estructuras sintácticas ni interpreta el significado completo. Es una aproximación con costos visibles.

## 7. Evaluar con la partición de prueba

**Pregunta.** ¿Cuántos de los 9 textos de prueba clasifica correctamente el método con negación?

**Operación.** Aplicamos el método a cada texto de prueba y comparamos con la etiqueta de referencia.

**Recordatorio.** La prueba es deliberadamente fácil y comparte vocabulario con el léxico. Sirve para comparar mecanismos, no para estimar rendimiento general.

In [ ]:
# loc con una condición se queda solo con las filas donde se cumple.
# copy() crea una copia, para no modificar el corpus original.
prueba = corpus.loc[corpus["partición"] == "prueba"].copy()

prueba["predicción"] = prueba["texto"].apply(predecir_con_negación)
prueba["acierto"] = prueba["etiqueta_referencia"] == prueba["predicción"]

display(prueba[["id", "texto", "etiqueta_referencia", "predicción", "acierto"]])

cantidad_de_aciertos = prueba["acierto"].sum()
print(f"Aciertos: {cantidad_de_aciertos} de {len(prueba)}")

**Los mismos resultados, en una matriz.** Las filas son la referencia y las columnas, la respuesta del método.

In [ ]:
dibujar_matriz_de_confusión(
    referencia=prueba["etiqueta_referencia"],
    predicción=prueba["predicción"],
    título="Prueba: léxico con negación",
)

**Leé la matriz.** Cada fila es la etiqueta de referencia y cada columna, lo que respondió el método. Los aciertos están en la diagonal; todo lo que esté fuera de ella es un error.

- **Observación:** los 9 textos de prueba se clasificaron correctamente.
- **Interpretación posible:** el léxico se armó con las mismas palabras que aparecen en el corpus, así que era esperable que la prueba saliera bien.
- **Límite:** 9 de 9 en nueve oraciones sencillas no permite afirmar que el método funcione con textos reales. Alcanza para comparar mecanismos, no para estimar rendimiento.

### ✎ Para pensar

1. Si cambiaras un valor del léxico, ¿cuántos textos de prueba podrían cambiar de etiqueta?
2. ¿Por qué un 100 % de aciertos puede ser una mala noticia para quien quiere aprender del método?

## 8. Los desafíos: dónde falla el método

**Pregunta.** ¿Qué pasa cuando los textos incluyen ironía, dos objetos de valoración, voces ajenas o uso coloquial?

**Operación.** Aplicamos el método a los 8 desafíos. No hay etiqueta de referencia automática: la lectura la hacés vos.

In [ ]:
revisión = desafíos.copy()
revisión["puntuación"] = revisión["texto"].apply(puntuación_con_negación)
revisión["predicción"] = revisión["puntuación"].apply(etiqueta_desde_puntuación)
display(revisión)

**Dos casos por dentro.** Miramos token por token la ironía (D03) y las valoraciones opuestas (D04).

In [ ]:
# Dos desafíos para mirar por dentro: la ironía (D03) y las valoraciones opuestas (D04).
dibujar_tokens_con_valor(
    tabla=contribuciones_con_negación("Buenísimo, se borró todo otra vez."),
    columna_valor="valor_final", título="D03: ironía",
)
dibujar_tokens_con_valor(
    tabla=contribuciones_con_negación("La profesora explicó bien, pero la plataforma funciona mal."),
    columna_valor="valor_final", título="D04: dos valoraciones que se anulan",
)

**Leé las tablas y los gráficos.**

| Desafío | Qué hizo el método | Qué muestra |
|---|---|---|
| D01 "no fue buena" | `NEG` | Lo resuelve la regla de negación. |
| D02 "no fue mala" | `POS` | Invierte el signo, aunque la lectura humana puede ser más matizada. |
| D03 ironía | `NEU` | `buenísimo` no está en el léxico: el método ni siquiera ve la palabra, y menos la ironía. |
| D04 dos objetos | `NEU` | +1 (`bien`) y −1 (`mal`) se anulan: se pierde que hay una valoración para la profesora y otra para la plataforma. |
| D05 voces diferentes | `NEG` | Cuenta `pésimo` (−2), que es lo que *dicen otros*, y no ve que la persona que escribe expresa lo contrario. |
| D06 evento desfavorable | `NEU` | El texto informa algo desfavorable sin usar palabras valorativas. |
| D07 uso coloquial | `POS` | Coincide con el uso de `buena` como favorable, pero "está buena" admite otras lecturas según el contexto. |
| D08 valoración mixta | `POS` | `buena` suma, pero `no resuelve mi problema` no aporta porque `resuelve` no está en el léxico. |

**Límite.** Un `NEU` puede significar ausencia de valoración (D06), compensación (D04) o falta de vocabulario (D03). La etiqueta no distingue entre las tres.

### ✎ Para pensar

1. ¿Qué desafío te parece más fácil de arreglar con una regla nueva? ¿Y cuál no se puede arreglar con reglas?
2. ¿Qué información perdemos cuando una oración tan rica como D04 se reduce a una sola etiqueta?

## 9. Experimento propio

**Producto.** Una modificación al analizador y una tabla que muestra qué cambió.

Elegí **una sola** modificación:

- Agregar una palabra al léxico.
- Cambiar la ventana de negación.
- Agregar un límite de alcance o un negador.

### ✎ Antes de modificar

Escribí:

1. Qué resultado esperás cambiar.
2. Por qué la modificación podría cambiarlo.
3. En qué caso podría perjudicar el resultado.

### Zona de experimento

Descomentá **una** línea (sacale el `#`) o escribí la tuya. Después ejecutá la celda, y a continuación la celda de comparación.

Si querés volver al léxico original, volvé a ejecutar la celda del léxico (sección 3).

In [ ]:
# Descomentá una línea o escribí la tuya:
# léxico["buenísimo"] = 1
# léxico["resuelve"] = 1
# VENTANA = 5
# NEGADORES.add("tampoco")
# LÍMITES.add("ni")

**Comparación.** Recalculamos con tu modificación y mostramos qué cambió.

In [ ]:
# Volvemos a calcular las predicciones con la configuración actual y comparamos con las anteriores.
revisión["predicción_nueva"] = revisión["texto"].apply(predecir_con_negación)
prueba["predicción_nueva"] = prueba["texto"].apply(predecir_con_negación)

cambiaron = revisión.loc[revisión["predicción"] != revisión["predicción_nueva"]]
aciertos_nuevos = (prueba["etiqueta_referencia"] == prueba["predicción_nueva"]).sum()

print(f"Desafíos que cambiaron de etiqueta: {len(cambiaron)} de {len(revisión)}")
print(f"Aciertos en la prueba: {aciertos_nuevos} de {len(prueba)}")
display(cambiaron[["id", "texto", "predicción", "predicción_nueva"]])

**Leé el resultado.**

- ¿Cambió lo que esperabas? ¿Cambió algo que no esperabas?
- ¿Mejoró algún caso a costa de empeorar otro?

**Atención metodológica.** Si usás los ejemplos de prueba para elegir modificaciones, pasan a ser ejemplos de desarrollo: ya no son una prueba independiente.

## Cierre

### Qué aprendimos

| Idea | Evidencia en este cuaderno |
|---|---|
| Cada decisión del método se puede inspeccionar. | Las fichas de colores muestran qué palabra aportó cuánto. |
| Una regla arregla un caso y abre otros. | La negación resolvió "no fue buena" pero falló con "No creo que… buena". |
| Un 0 no significa lo mismo en todos los casos. | D03, D04 y D06 dan `NEU` por tres razones distintas. |
| Un 100 % de aciertos depende de qué se evalúa. | 9 de 9 en la prueba y varios errores en los desafíos. |

### Limitaciones del método

- El léxico es finito y fue armado a mano: lo que no está, no cuenta.
- Las reglas no entienden el contexto, la ironía ni quién habla.
- Los valores numéricos son una convención, no una medición.

### Conclusión: completá con tus palabras

- El criterio del método proviene de...
- La puntuación se obtiene mediante...
- Una regla que funcionó en estos ejemplos fue...
- Un error o limitación fue...
- La evidencia textual que sostiene mi explicación es...
- Estos resultados no permiten afirmar...

### Conexión con el próximo cuaderno

Acá el léxico y las reglas los escribió una persona. En el cuaderno **02** el programa va a *aprender* qué palabras se asocian a cada etiqueta a partir de ejemplos, y vamos a discutir qué significa que un método "aprenda" de un corpus pequeño.